### Imports

In [1]:
import os
from pathlib import Path
from pickle import load
from time import time

import numpy as np
import pandas as pd
import pyspark
from ase.atoms import Atoms
from ase.io import iread, read
from colabfit.tools.property_definitions import (
    atomic_forces_pd,
    cauchy_stress_pd,
    energy_pd,
)
from colabfit.tools.vast.configuration import AtomicConfiguration  # noqa
from colabfit.tools.vast.database import DataManager, VastDataLoader, generate_ds_id
from colabfit.tools.vast.property import PropertyMap, property_info
from dotenv import load_dotenv
from pyspark.sql import SparkSession
from tqdm import tqdm

### Info about ChIMES

 Chebyshev Interaction Model for Efficient Simulation

In [2]:
# generate_ds_id()

In [3]:
"""
From `https://github.com/rk-lindsey/chimes_calculator/blob/3b9e38bc7016c3af69e140155395dcd1ad854d10/chimesFF/examples/c/main.c#L341`
xyzf configuration header format:
lx,ly,lz,stress[0],stress[1],stress[2],stress[3],stress[4],stress[5],stress[6],stress[7],stress[8],epot

But it appears to me to be NONORTH 9 for cell, 6 for stress, 1 for epot

"""

'\nFrom `https://github.com/rk-lindsey/chimes_calculator/blob/3b9e38bc7016c3af69e140155395dcd1ad854d10/chimesFF/examples/c/main.c#L341`\nxyzf configuration header format:\nlx,ly,lz,stress[0],stress[1],stress[2],stress[3],stress[4],stress[5],stress[6],stress[7],stress[8],epot\n\nBut it appears to me to be NONORTH 9 for cell, 6 for stress, 1 for epot\n\n'

In [4]:
"""
options for header
NON_ORTHO:
        cell                                    stress                  epot
10  lx ly lz                      s0 s1 s2 s3 s4 s5            e1
9   lx ly lz                      s0 s1 s2 s3 s4 s5
7   lx ly lz                      s0 s1 s2                     e1
6   lx ly lz                      s0 s1 s2
4   lx ly lz                                                   e1

17  NON_ORTHO l0 l1 l2 l3 l4 l5 l6 l7 l8    s0 s1 s2 s3 s4 s5            e1
16  NON_ORTHO l0 l1 l2 l3 l4 l5 l6 l7 l8    s0 s1 s2 s3 s4 s5
14  NON_ORTHO l0 l1 l2 l3 l4 l5 l6 l7 l8    s0 s1 s2                     e1
13  NON_ORTHO l0 l1 l2 l3 l4 l5 l6 l7 l8    s0 s1 s2

11  NON_ORTHO l0 l1 l2 l3 l4 l5 l6 l7 l8                                 e1
10  NON_ORTHO l0 l1 l2 l3 l4 l5 l6 l7 l8

"""

'\noptions for header\nNON_ORTHO:\n        cell                                    stress                  epot\n10  lx ly lz                      s0 s1 s2 s3 s4 s5            e1\n9   lx ly lz                      s0 s1 s2 s3 s4 s5\n7   lx ly lz                      s0 s1 s2                     e1\n6   lx ly lz                      s0 s1 s2\n4   lx ly lz                                                   e1\n\n17  NON_ORTHO l0 l1 l2 l3 l4 l5 l6 l7 l8    s0 s1 s2 s3 s4 s5            e1\n16  NON_ORTHO l0 l1 l2 l3 l4 l5 l6 l7 l8    s0 s1 s2 s3 s4 s5\n14  NON_ORTHO l0 l1 l2 l3 l4 l5 l6 l7 l8    s0 s1 s2                     e1\n13  NON_ORTHO l0 l1 l2 l3 l4 l5 l6 l7 l8    s0 s1 s2\n\n11  NON_ORTHO l0 l1 l2 l3 l4 l5 l6 l7 l8                                 e1\n10  NON_ORTHO l0 l1 l2 l3 l4 l5 l6 l7 l8\n\n'

In [5]:
# header_ex = "NON_ORTHO 10.349112000 0.000000000 0.000000000 0.000000000 10.349112000 0.000000000 0.000000000 0.000000000 10.349112000 119.71296400000001 107.753755 126.75542399999999 -6.632839 -1.79269 -0.12439599999999999 -30725.995371049197"
# header_ex = "NON_ORTHO 10.349112000 0.000000000 0.000000000 0.000000000 10.349112000 0.000000000 0.000000000 0.000000000 10.349112000 119.71296400000001 107.753755 126.75542399999999 -6.632839 -1.79269 -0.12439599999999999"
# header_ex = "NON_ORTHO 10.349112000 0.000000000 0.000000000 0.000000000 10.349112000 0.000000000 0.000000000 0.000000000 10.349112000 119.71296400000001 107.753755 126.75542399999999 -30725.995371049197"
# header_ex = "NON_ORTHO 10.349112000 0.000000000 0.000000000 0.000000000 10.349112000 0.000000000 0.000000000 0.000000000 10.349112000 119.71296400000001 107.753755 126.75542399999999"
# header_ex = "NON_ORTHO 10.349112000 0.000000000 0.000000000 0.000000000 10.349112000 0.000000000 0.000000000 0.000000000 10.349112000 -30725.995371049197"
# header_ex = "NON_ORTHO 10.349112000 0.000000000 0.000000000 0.000000000 10.349112000 0.000000000 0.000000000 0.000000000 10.349112000"
# header_ex = "10.349112000 10.349112000 10.349112000 119.71296400000001 107.753755 126.75542399999999 -6.632839 -1.79269 -0.12439599999999999 -30725.995371049197"
# header_ex = "10.349112000 10.349112000 10.349112000 119.71296400000001 107.753755 126.75542399999999 -6.632839 -1.79269 -0.12439599999999999"
# header_ex = "10.349112000 10.349112000 10.349112000 119.71296400000001 107.753755 126.75542399999999 -30725.995371049197"
# header_ex = "10.349112000 10.349112000 10.349112000 119.71296400000001 107.753755 126.75542399999999"
header_ex = "10.349112000 10.349112000 10.349112000 -30725.995371049197"

### ChIMES functions

In [2]:
def get_pbc_from_cell(cell):
    cell = np.array(cell, dtype=float)
    cell_lengths = np.linalg.norm(cell, axis=1)
    pbc = cell_lengths > 1e-10
    return pbc.astype(bool).tolist()


def chimes_md_map_from_file(fp):
    try:
        df = pd.read_excel(fp, skiprows=2, header=0)
        cols = df.columns
        cols = [col.lower().replace(" ", "_") for col in cols]
        assert "first_frame" in cols and "last_frame" in cols
        df.columns = cols
    except Exception as e:
        print(f"Error reading Excel file: {e}")
        return None
    md_map = {
        (row["first_frame"], row["last_frame"]): row.to_dict()
        for _, row in df.iterrows()
    }
    return md_map


def chimes_get_metadata_by_ix(index, md_map):
    """md_map keys should be tuples of (first_frame, last_frame)"""
    keys = list(md_map)
    key_bool = [x[0] <= index <= x[1] for x in keys]
    key = keys[key_bool.index(True)]
    return md_map[key]


def chimes_header_parser(headerline):
    """Parse ChIMES header, return cell, stress, and epot."""
    parts = [x.strip() for x in headerline.split()]
    parts_len = len(parts)
    stress = None
    epot = None
    if parts[0] == "NON_ORTHO":
        cell = np.array(parts[1:10], dtype=float).reshape(3, 3)
        if parts_len >= 16:
            stress = stress_to_9_tensor(np.array(parts[10:16], dtype=float))
            if parts_len == 17:
                epot = float(parts[-1])
        elif parts_len >= 13:
            stress = stress_to_9_tensor(np.array(parts[10:13], dtype=float))
            if parts_len == 14:
                epot = float(parts[-1])
        elif parts_len == 11:
            epot = float(parts[-1])
        else:
            raise ValueError(f"Unexpected length of parts: {parts_len}: {parts}")
    else:
        cell = np.diag(np.array([float(x) for x in parts[0:3]]))
        if 9 <= parts_len <= 10:
            stress = stress_to_9_tensor(np.array(parts[3:9], dtype=float))
            if parts_len == 10:
                epot = float(parts[-1])
        elif 6 <= parts_len <= 7:
            stress = stress_to_9_tensor(np.array(parts[3:6], dtype=float))
            if parts_len == 7:
                epot = float(parts[-1])
        elif parts_len == 4:
            epot = float(parts[-1])
        else:
            raise ValueError(f"Unexpected length of parts: {parts_len}: {parts}")
    pbc = get_pbc_from_cell(cell)
    return {
        "Lattice": cell,
        "stress": stress,
        "epot": epot,
        "pbc": pbc,
        "Properties": "species:S:1:pos:R:3:forces:R:3",
    }


def stress_to_9_tensor(stress):
    if len(stress) == 6:
        xx, yy, zz, xy, xz, yz = stress
        return np.array([[xx, xy, xz], [xy, yy, yz], [xz, yz, zz]], dtype=float)
    elif len(stress) == 3:
        xx, yy, zz = stress
        return np.array([[xx, 0.0, 0.0], [0.0, yy, 0.0], [0.0, 0.0, zz]], dtype=float)
    else:
        raise ValueError(f"Unexpected length of stress: {len(stress)}: {stress}")


# header_parser(header_ex)

### Testing functions

In [3]:
xls = "data/ChIMES_C-2.0_Small_model_data_defs.xls"
xyz = Path("ChIMES_C-2.0.Small_model_dataset.xyzf")

In [4]:
CO_MD_MAP = {key: {"field": key} for key in ["phase", "t_(k)", "class"]}

In [5]:
def read_chimes(xyz, xls):
    for i, atom in tqdm(
        enumerate(
            iread(
                xyz, format="extxyz", properties_parser=chimes_header_parser, index=":"
            )
        )
    ):
        atom.info.update(chimes_get_metadata_by_ix(i, chimes_md_map_from_file(xls)))
        atom.info["_name"] = (
            f"{xyz.stem}__frame_range_{atom.info['first_frame']}-{atom.info['last_frame']}__index_{i}"
        )
        yield AtomicConfiguration.from_ase(atoms=atom, co_md_map=CO_MD_MAP)


xyz = Path("data/ChIMES_C-2.0.Small_model_dataset.xyzf")
chimes_gen = read_chimes(xyz, xls)

In [6]:
property_keys = [
    "cumul_frames",
    "labeling_method",
    "real_alc",
    "eff._alc",
    "case",
    "good/bad",
    "t_(k)",
    "rho_(gcc)",
    "class",
    "total_frames",
    "first_frame",
    "last_frame",
    "data_source",
    "path",
]
property_map = PropertyMap([energy_pd, atomic_forces_pd, cauchy_stress_pd])
property_map.set_metadata_field("software", "ChIMES")
property_map.set_metadata_field("method", "DFT-PBE")
property_map.set_metadata_field("input", "some input")
for property in property_keys:
    property_map.set_metadata_field(property, property, dynamic=True)
energy_info = property_info(
    property_name="energy",
    field="epot",
    units="kcal/mol",
    original_file_key="in header",
    additional=[("per-atom", {"value": False, "units": None})],
)
force_info = property_info(
    property_name="atomic-forces",
    field="forces",
    units="hartree/bohr",
    original_file_key="forces",
    additional=None,
)
stress_info = property_info(
    property_name="cauchy-stress",
    field="stress",
    units="GPa",
    original_file_key="in header",
    additional=[("volume-normalized", {"value": True, "units": None})],
)
property_map.set_properties([energy_info, force_info, stress_info])
PROPERTY_MAP = property_map.get_property_map()

In [16]:
dm = DataManager(
    prop_defs=[energy_pd, atomic_forces_pd, cauchy_stress_pd],
    configs=chimes_gen,
    prop_map=property_map.get_property_map(),
    dataset_id="DS_2d6k0sqwxfhj_0",
)

Dataset ID: DS_2d6k0sqwxfhj_0


In [17]:
mdmap = chimes_md_map_from_file(xls)

In [18]:
cs_names = [
    f"{'test_name'}__{row['phase']}_phase__temp_{row['t_(k)']}__frame_range_{row['first_frame']}-{row['last_frame']}"
    for row in mdmap.values()
]

In [19]:
DATASET_NAME = "ChIMES_C_2.0-Small_2025"

In [20]:
cs_name_match = [
    f"frame_range_{row['first_frame']}-{row['last_frame']}" for row in mdmap.values()
]
cs_descriptions = [
    f"Configurations from {DATASET_NAME}, {row['phase']} phase at {row['t_(k)']} K, of class \"{row['class']}\" frame range {row['first_frame']}-{row['last_frame']}"
    for row in mdmap.values()
]
cs_descriptions

['Configurations from ChIMES_C_2.0-Small_2025, diamond phase at 3000 K, of class "0" frame range 0-24',
 'Configurations from ChIMES_C_2.0-Small_2025, graphite phase at 3000 K, of class "1" frame range 25-49',
 'Configurations from ChIMES_C_2.0-Small_2025, liquid phase at 1000 K, of class "2" frame range 50-74',
 'Configurations from ChIMES_C_2.0-Small_2025, liquid phase at 2000 K, of class "3" frame range 75-99',
 'Configurations from ChIMES_C_2.0-Small_2025, liquid phase at 6000 K, of class "4" frame range 100-124',
 'Configurations from ChIMES_C_2.0-Small_2025, liquid phase at 6000 K, of class "5" frame range 125-149',
 'Configurations from ChIMES_C_2.0-Small_2025, liquid phase at 8000 K, of class "6" frame range 150-174',
 'Configurations from ChIMES_C_2.0-Small_2025, diamond phase at 300 K, of class "7" frame range 175-199',
 'Configurations from ChIMES_C_2.0-Small_2025, graphite? phase at 300 K, of class "8" frame range 200-224',
 'Configurations from ChIMES_C_2.0-Small_2025, gra

In [21]:
atom1, (co1, po1) = dm.co_po_example_rows()

2it [01:40, 53.14s/it]

{'phase': 'diamond', 't_(k)': 3000, 'class': 0}
{"class": 0, "phase": "diamond", "t_(k)": 3000, "hash": "8908055112069523572346736107319131021312437592095156622463186729691212419526199926669591990246754271842015887009852535439286968679114566776490287281541969", "id": "MD_8908055112069523572346736"}
{'input': {'source-value': 'some input'}, 'property_keys': {'source-value': {'energy': 'in header', 'atomic-forces': 'forces', 'cauchy-stress': 'in header'}}, 'cumul_frames': {'source-value': 25}, 'labeling_method': {'source-value': 'PBE'}, 'real_alc': {'source-value': 1.0}, 'eff._alc': {'source-value': 1.0}, 'case': {'source-value': 0}, 'good/bad': {'source-value': nan}, 't_(k)': {'source-value': 3000}, 'rho_(gcc)': {'source-value': 3.67}, 'class': {'source-value': 0}, 'total_frames': {'source-value': 25}, 'first_frame': {'source-value': 0}, 'last_frame': {'source-value': 24}, 'data_source': {'source-value': 'DFT'}, 'path': {'source-value': '/p/lustre2/rlindsey/ChIMES_Carbon-210317/generate

In [30]:
co1

{'id': 'CO_1165574006080314956046330',
 'hash': '11655740060803149560463306529714150188308495514113511243224301537464891346748521231025726401572265012459204567246815925665480730103051308100765116622761077',
 'last_modified': datetime.datetime(2025, 3, 31, 17, 47, 25, tzinfo=tzutc()),
 'dataset_ids': ['DS_2d6k0sqwxfhj_0'],
 'chemical_formula_hill': 'C216',
 'chemical_formula_reduced': 'C',
 'chemical_formula_anonymous': 'A',
 'elements': ['C'],
 'elements_ratios': [1.0],
 'atomic_numbers': [6,
  6,
  6,
  6,
  6,
  6,
  6,
  6,
  6,
  6,
  6,
  6,
  6,
  6,
  6,
  6,
  6,
  6,
  6,
  6,
  6,
  6,
  6,
  6,
  6,
  6,
  6,
  6,
  6,
  6,
  6,
  6,
  6,
  6,
  6,
  6,
  6,
  6,
  6,
  6,
  6,
  6,
  6,
  6,
  6,
  6,
  6,
  6,
  6,
  6,
  6,
  6,
  6,
  6,
  6,
  6,
  6,
  6,
  6,
  6,
  6,
  6,
  6,
  6,
  6,
  6,
  6,
  6,
  6,
  6,
  6,
  6,
  6,
  6,
  6,
  6,
  6,
  6,
  6,
  6,
  6,
  6,
  6,
  6,
  6,
  6,
  6,
  6,
  6,
  6,
  6,
  6,
  6,
  6,
  6,
  6,
  6,
  6,
  6,
  6,
  6,
  

In [29]:
atom1.info

{'epot': -45178.7287943478,
 'cumul_frames': 25,
 'labeling_method': 'PBE',
 'real_alc': 1.0,
 'eff._alc': 1.0,
 'case': 0,
 'good/bad': nan,
 'phase': 'diamond',
 't_(k)': 3000,
 'rho_(gcc)': 3.67,
 'class': 0,
 'total_frames': 25,
 'first_frame': 0,
 'last_frame': 24,
 'data_source': 'DFT',
 'path': '/p/lustre2/rlindsey/ChIMES_Carbon-210317/generate_models-DFTBref/al_2+3+4b-O20_10_4-postMioPBE/ALC-2/refit_with_O20-10-4_rc_4.5_3.125_3.125-noVDW/all.novdw.OUTCAR-fixed-reDFT.xyzf ',
 '_name': 'ChIMES_C-2.0.Small_model_dataset__frame_range_0-24__index_2'}

In [22]:
po1

{'id': 'PO_3356190005488905594442261',
 'hash': '335619000548890559444226126499950355566299024920169660687469379658426701364333959556648888185326989479470279611743653684824591254646984033212384478108825',
 'last_modified': datetime.datetime(2025, 3, 31, 17, 47, 25, tzinfo=tzutc()),
 'configuration_id': 'CO_1165574006080314956046330',
 'dataset_id': 'DS_2d6k0sqwxfhj_0',
 'multiplicity': 1,
 'metadata_id': 'MD_9578825250603084140472001',
 'metadata_path': '2001/MD_9578825250603084140472001.json',
 'metadata_size': 799,
 'software': 'ChIMES',
 'method': 'DFT-PBE',
 'chemical_formula_hill': 'C216',
 'energy': -1959.135089534821,
 'atomic_forces_00': [[-0.10303449020207033,
   0.061200291169581915,
   -0.06633731560974764],
  [0.20080495535975046, 0.2583382290829458, 0.21793603686344504],
  [0.044034209499368794, 0.1612967673936092, 0.04250720223440227],
  [0.03531716802673407, 0.07581136068394077, 0.0060210286459485745],
  [0.057839275179043276, 0.229737093008743, 0.0029580140731964594],
 